In [1]:
import os
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

try:
    from imblearn.over_sampling import RandomOverSampler

    IMBLEARN_AVAILABLE = True
except ImportError:
    IMBLEARN_AVAILABLE = False

try:
    from xgboost import XGBClassifier

    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False



# KONFIGURÁCIA


try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

DATASET_PATH = BASE_DIR / "malicious_phish.csv"
OUTPUT_DIR = BASE_DIR / "bakalarka_outputs"

URL_COL = "url"
LABEL_COL = "type"
RANDOM_STATE = 42
TEST_SIZE = 0.20
USE_OVERSAMPLING = False
USE_MI = True
CORRELATION_THRESHOLD = 0.90
TOP_K = 20



# POMOCNÉ FUNKCIE


def sanitize_filename(name: str) -> str:
    return (
        name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace("(", "")
        .replace(")", "")
    )


def count_query_params(query: str) -> int:
    if not query:
        return 0
    return len([part for part in query.split("&") if "=" in part and part.strip()])


def safe_divide(numerator, denominator):
    return np.divide(numerator, denominator.replace(0, 1))



# NAČÍTANIE DÁT


def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dataset sa nenašiel: {path}")

    df = pd.read_csv(path)
    df = df.drop(columns=["Unnamed: 0"], errors="ignore")

    required_cols = {URL_COL, LABEL_COL}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"V datasete chýbajú stĺpce: {missing_cols}")

    df = df[[URL_COL, LABEL_COL]].copy()

    df[URL_COL] = df[URL_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

    df = df[df[URL_COL].str.len() > 0]
    df = df.drop_duplicates(subset=[URL_COL]).reset_index(drop=True)

    return df



# FEATURE ENGINEERING


def extract_features(urls: pd.Series) -> pd.DataFrame:
    t0 = time.perf_counter()

    original_urls = urls.astype(str).str.strip()
    u = original_urls.str.lower()
    u_no_proto = u.str.replace(r"^https?://", "", regex=True)

    domain = u_no_proto.str.split("/", n=1).str[0].fillna("")
    domain_no_port = domain.str.split(":", n=1).str[0].fillna("")

    path_query = u_no_proto.str.split("/", n=1).str[1].fillna("")
    path = path_query.str.split(r"\?", n=1, regex=True).str[0].fillna("")
    query = path_query.str.split(r"\?", n=1, regex=True).str[1].fillna("")

    tld = domain_no_port.str.split(".").str[-1].fillna("")
    domain_without_tld = domain_no_port.str.rsplit(".", n=1).str[0].fillna(domain_no_port)

    tokens = u.str.split(r"[\/\.\-\_\?\=\&]+", regex=True)

    token_counts = tokens.map(lambda parts: len([part for part in parts if part]))
    avg_token_len = tokens.map(
        lambda parts: np.mean([len(part) for part in parts if part]) if any(parts) else 0.0
    )
    max_token_len = tokens.map(
        lambda parts: max((len(part) for part in parts if part), default=0)
    )

    ip_regex = r"(?:\d{1,3}\.){3}\d{1,3}"
    shorteners_regex = r"(?:bit\.ly|tinyurl\.com|t\.co|goo\.gl|ow\.ly|is\.gd)"
    suspicious_words_regex = (
        r"(?:login|verify|signin|bank|secure|account|update|password|confirm|wallet)"
    )
    suspicious_tlds = {
        "zip",
        "xyz",
        "top",
        "click",
        "work",
        "support",
        "review",
        "country",
        "stream",
    }

    url_len = u.str.len()
    domain_len = domain_no_port.str.len()
    path_len = path.str.len()
    query_len = query.str.len()

    count_digits = u.str.count(r"\d")
    count_special = u.str.count(r"[@\?=&%#]")
    count_uppercase = original_urls.str.count(r"[A-Z]")
    count_letters = u.str.count(r"[a-z]")
    count_percent = u.str.count(r"%")
    count_tilde = u.str.count(r"~")
    count_semicolon = u.str.count(r";")
    count_colon = u.str.count(r":")
    count_double_slash = u.str.count(r"//")
    count_path_segments = path.str.count(r"/") + (path.str.len() > 0).astype(int)

    repeated_special_regex = r"([\-_=@%])\1+"
    encoded_regex = r"%[0-9a-f]{2}"
    hex_like_regex = r"(?:0x[0-9a-f]+)"

    features = pd.DataFrame(
        {
            "url_len": url_len,
            "domain_len": domain_len,
            "path_len": path_len,
            "query_len": query_len,
            "domain_without_tld_len": domain_without_tld.str.len(),
            "count_dots": u.str.count(r"\."),
            "count_slashes": u.str.count(r"/"),
            "count_hyphens": u.str.count(r"-"),
            "count_underscores": u.str.count(r"_"),
            "count_digits": count_digits,
            "count_letters": count_letters,
            "count_special": count_special,
            "count_percent": count_percent,
            "count_tilde": count_tilde,
            "count_semicolon": count_semicolon,
            "count_colon": count_colon,
            "count_double_slash": count_double_slash,
            "count_params": query.apply(count_query_params),
            "count_uppercase": count_uppercase,
            "count_path_segments": count_path_segments,
            "uses_https": u.str.startswith("https://").astype(int),
            "has_ip": domain_no_port.str.contains(ip_regex, regex=True).astype(int),
            "num_subdomains": (domain_no_port.str.count(r"\.") - 1).clip(lower=0),
            "is_shortened": domain_no_port.str.contains(shorteners_regex, regex=True).astype(int),
            "has_at": u.str.contains("@", regex=False).astype(int),
            "has_www": domain_no_port.str.startswith("www.").astype(int),
            "has_login": u.str.contains("login", regex=False).astype(int),
            "has_secure": u.str.contains("secure", regex=False).astype(int),
            "has_account": u.str.contains("account", regex=False).astype(int),
            "has_update": u.str.contains("update", regex=False).astype(int),
            "has_verify": u.str.contains("verify", regex=False).astype(int),
            "has_bank": u.str.contains("bank", regex=False).astype(int),
            "has_password": u.str.contains("password", regex=False).astype(int),
            "domain_has_hyphen": domain_no_port.str.contains("-", regex=False).astype(int),
            "path_has_login": path.str.contains("login", regex=False).astype(int),
            "path_has_verify": path.str.contains("verify", regex=False).astype(int),
            "path_has_bank": path.str.contains("bank", regex=False).astype(int),
            "tld_is_suspicious": tld.isin(suspicious_tlds).astype(int),
            "has_suspicious_words": u.str.contains(suspicious_words_regex, regex=True).astype(int),
            "has_encoded_chars": u.str.contains(encoded_regex, regex=True).astype(int),
            "has_repeated_special": u.str.contains(repeated_special_regex, regex=True).astype(int),
            "has_hex_like_pattern": u.str.contains(hex_like_regex, regex=True).astype(int),
            "num_tokens_url": token_counts,
            "avg_token_len": avg_token_len,
            "max_token_len": max_token_len,
            "digit_ratio": safe_divide(count_digits, url_len),
            "special_ratio": safe_divide(count_special, url_len),
            "uppercase_ratio": safe_divide(count_uppercase, url_len),
            "letter_ratio": safe_divide(count_letters, url_len),
            "domain_digit_ratio": safe_divide(domain_no_port.str.count(r"\d"), domain_len),
        }
    )

    print(f"Extrakcia príznakov trvala: {time.perf_counter() - t0:.2f} s")
    print("X shape:", features.shape)
    return features



# ANALÝZA PRÍZNAKOV


def save_numeric_feature_statistics(X: pd.DataFrame, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    stats = X.describe().T
    stats["median"] = X.median()
    stats["variance"] = X.var()
    stats["missing_values"] = X.isna().sum()
    stats = stats[
        ["count", "mean", "std", "min", "25%", "50%", "75%", "max", "median", "variance", "missing_values"]
    ]

    stats_path = output_dir / "numeric_feature_statistics.csv"
    stats.to_csv(stats_path, encoding="utf-8")
    print(f"Štatistické ukazovatele príznakov uložené do: {stats_path}")


def identify_binary_and_numeric_features(X: pd.DataFrame) -> Tuple[List[str], List[str]]:
    binary_features = []
    numeric_features = []

    for col in X.columns:
        unique_values = set(pd.Series(X[col]).dropna().unique().tolist())
        if unique_values.issubset({0, 1}) and len(unique_values) <= 2:
            binary_features.append(col)
        else:
            numeric_features.append(col)

    return binary_features, numeric_features


def save_binary_feature_frequencies(X: pd.DataFrame, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    binary_features, _ = identify_binary_and_numeric_features(X)

    rows = []
    for col in binary_features:
        counts = X[col].value_counts(dropna=False).to_dict()
        rows.append(
            {
                "feature": col,
                "count_0_absent": int(counts.get(0, 0)),
                "count_1_present": int(counts.get(1, 0)),
                "present_ratio": float(counts.get(1, 0) / len(X)),
                "absent_ratio": float(counts.get(0, 0) / len(X)),
            }
        )

    freq_df = pd.DataFrame(rows).sort_values(by="count_1_present", ascending=False)
    freq_path = output_dir / "binary_feature_frequencies.csv"
    freq_df.to_csv(freq_path, index=False, encoding="utf-8")
    print(f"Početnosti binárnych príznakov uložené do: {freq_path}")


def plot_top_binary_feature_frequencies(X: pd.DataFrame, output_dir: Path, top_n: int = 15) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    binary_features, _ = identify_binary_and_numeric_features(X)
    if not binary_features:
        return

    counts = []
    for col in binary_features:
        present_count = int(X[col].sum())
        absent_count = int(len(X) - present_count)
        counts.append((col, present_count, absent_count))

    counts_sorted = sorted(counts, key=lambda x: x[1], reverse=True)[:top_n]

    feature_names = [x[0] for x in counts_sorted]
    present_values = [x[1] for x in counts_sorted]
    absent_values = [x[2] for x in counts_sorted]

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(feature_names, absent_values, label="0 - neprítomnosť")
    ax.barh(feature_names, present_values, left=absent_values, label="1 - prítomnosť")
    ax.set_xlabel("Počet výskytov")
    ax.set_title(f"Najlepších {top_n} binárnych príznakov podľa prítomnosti")
    ax.legend()
    fig.tight_layout()

    plot_path = output_dir / "top_binary_feature_frequencies.png"
    fig.savefig(plot_path, dpi=200)
    plt.close(fig)
    print(f"Graf binárnych príznakov uložený do: {plot_path}")


def plot_selected_numeric_histograms(X: pd.DataFrame, output_dir: Path, features: List[str]) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    for col in features:
        if col not in X.columns:
            continue

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(X[col], bins=40)
        ax.set_title(f"Histogram príznaku: {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Počet")
        fig.tight_layout()

        plot_path = output_dir / f"hist_{sanitize_filename(col)}.png"
        fig.savefig(plot_path, dpi=200)
        plt.close(fig)


def save_full_correlation_matrix(X: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)

    corr = X.corr()
    corr_path = output_dir / "full_correlation_matrix.csv"
    corr.to_csv(corr_path, encoding="utf-8")
    print(f"Celá korelačná matica uložená do: {corr_path}")

    return corr


def save_correlation_heatmap(corr: pd.DataFrame, output_dir: Path, file_name: str) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(16, 14))
    im = ax.imshow(corr.values, aspect="auto")
    ax.set_title("Korelačná matica príznakov")
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(corr.columns, fontsize=7)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()

    heatmap_path = output_dir / file_name
    fig.savefig(heatmap_path, dpi=220)
    plt.close(fig)
    print(f"Heatmapa korelačnej matice uložená do: {heatmap_path}")


def save_high_correlation_pairs(X: pd.DataFrame, output_dir: Path, threshold: float = 0.90) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    pairs = []
    for col in upper.columns:
        for row in upper.index:
            value = upper.loc[row, col]
            if pd.notna(value) and value >= threshold:
                pairs.append({"feature_1": row, "feature_2": col, "abs_correlation": value})

    pairs_df = pd.DataFrame(pairs).sort_values(by="abs_correlation", ascending=False)
    pairs_path = output_dir / "high_correlation_pairs.csv"
    pairs_df.to_csv(pairs_path, index=False, encoding="utf-8")
    print(f"Silno korelované páry príznakov uložené do: {pairs_path}")



# FEATURE SELECTION


def correlation_filter(
    X: pd.DataFrame, threshold: float = 0.90
) -> Tuple[pd.DataFrame, List[str]]:
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return X.drop(columns=to_drop), to_drop


def select_features(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: np.ndarray,
    top_k: int,
    use_mi: bool,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, List[str], List[str], str]:
    X_train_cf, dropped = correlation_filter(X_train, threshold=CORRELATION_THRESHOLD)
    X_test_cf = X_test[X_train_cf.columns].copy()

    effective_top_k = min(top_k, X_train_cf.shape[1])

    if use_mi:
        scores = mutual_info_classif(X_train_cf, y_train, random_state=RANDOM_STATE)
        score_name = "mutual_info"
    else:
        scores, _ = chi2(X_train_cf, y_train)
        score_name = "chi2"

    score_series = pd.Series(scores, index=X_train_cf.columns).sort_values(ascending=False)
    selected_features = score_series.head(effective_top_k).index.tolist()

    print("Odstránené kvôli korelácii:", dropped)
    print(f"Top-{effective_top_k} features podľa {score_name}:")
    print(score_series.head(effective_top_k))

    return (
        X_train_cf[selected_features].copy(),
        X_test_cf[selected_features].copy(),
        score_series,
        dropped,
        selected_features,
        score_name,
    )



# MODELY


def get_models(is_binary: bool) -> Dict[str, object]:
    models: Dict[str, object] = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "LinearSVC": LinearSVC(random_state=RANDOM_STATE, dual=False, max_iter=5000),
    }

    if XGBOOST_AVAILABLE:
        if is_binary:
            models["XGBoost"] = XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        else:
            models["XGBoost"] = XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
    else:
        print("XGBoost nie je nainštalovaný. Model XGBoost sa preskočí.")

    return models


def get_scores(model, X) -> Optional[np.ndarray]:
    if hasattr(model, "predict_proba"):
        probas = model.predict_proba(X)
        if probas.ndim == 2 and probas.shape[1] >= 2:
            return probas[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None



# UKLADANIE GRAFOV


def save_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: List[str],
    title: str,
    output_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    label_ids = list(range(len(class_names)))
    cm = confusion_matrix(y_true, y_pred, labels=label_ids)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)


def save_single_binary_roc(
    y_true: np.ndarray,
    scores: np.ndarray,
    model_name: str,
    output_path: Path,
) -> float:
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, label=f"{model_name} (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC krivka - {model_name}")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)

    return auc


def save_binary_roc_comparison(
    y_true: np.ndarray,
    score_map: Dict[str, np.ndarray],
    output_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))

    for model_name, scores in score_map.items():
        fpr, tpr, _ = roc_curve(y_true, scores)
        auc = roc_auc_score(y_true, scores)
        ax.plot(fpr, tpr, label=f"{model_name} (AUC = {auc:.4f})")

    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC krivky - porovnanie modelov")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)



# VYHODNOTENIE


def evaluate_task(
    task_name: str,
    X: pd.DataFrame,
    y: np.ndarray,
    class_names: List[str],
    is_binary: bool,
    output_dir: Path,
) -> pd.DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"Úloha: {task_name}")
    print("=" * 80)

    task_dir = output_dir / sanitize_filename(task_name)
    cm_dir = task_dir / "confusion_matrices"
    roc_dir = task_dir / "roc_curves"
    task_dir.mkdir(parents=True, exist_ok=True)
    cm_dir.mkdir(parents=True, exist_ok=True)
    roc_dir.mkdir(parents=True, exist_ok=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    if USE_OVERSAMPLING:
        if not IMBLEARN_AVAILABLE:
            raise ImportError("Chýba imbalanced-learn. Nainštaluj: pip install imbalanced-learn")
        ros = RandomOverSampler(random_state=RANDOM_STATE)
        X_train, y_train = ros.fit_resample(X_train, y_train)
        print("Po oversamplingu X_train:", X_train.shape)

    (
        X_train_sel,
        X_test_sel,
        score_series,
        dropped_features,
        selected_features,
        score_name,
    ) = select_features(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        top_k=TOP_K,
        use_mi=USE_MI,
    )

    feature_scores_path = task_dir / f"{sanitize_filename(task_name)}_feature_scores.csv"
    score_series.to_csv(feature_scores_path, header=[score_name])

    selected_features_path = task_dir / f"{sanitize_filename(task_name)}_selected_features.txt"
    with open(selected_features_path, "w", encoding="utf-8") as f:
        f.write(f"Task: {task_name}\n")
        f.write(f"Score method: {score_name}\n")
        f.write(f"Correlation threshold: {CORRELATION_THRESHOLD}\n")
        f.write(f"Top-K: {TOP_K}\n\n")
        f.write("Dropped due to correlation:\n")
        for item in dropped_features:
            f.write(f"- {item}\n")
        f.write("\nSelected features:\n")
        for item in selected_features:
            f.write(f"- {item}\n")

    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_test_scaled = scaler.transform(X_test_sel)

    models = get_models(is_binary=is_binary)
    results = []
    roc_score_map: Dict[str, np.ndarray] = {}

    label_ids = list(range(len(class_names)))

    for name, model in models.items():
        t0 = time.perf_counter()

        if name in {"Logistic Regression", "LinearSVC"}:
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_test_scaled)
            score_values = get_scores(model, X_test_scaled)
        else:
            model.fit(X_train_sel, y_train)
            preds = model.predict(X_test_sel)
            score_values = get_scores(model, X_test_sel)

        elapsed = time.perf_counter() - t0
        average_type = "binary" if is_binary else "weighted"

        acc = accuracy_score(y_test, preds)
        precision = precision_score(
            y_test,
            preds,
            average=average_type,
            zero_division=0,
        )
        recall = recall_score(
            y_test,
            preds,
            average=average_type,
            zero_division=0,
        )
        f1 = f1_score(
            y_test,
            preds,
            average=average_type,
            zero_division=0,
        )

        roc_auc = np.nan
        if is_binary and score_values is not None:
            roc_auc = roc_auc_score(y_test, score_values)
            roc_score_map[name] = score_values

        results.append(
            {
                "Task": task_name,
                "Model": name,
                "Accuracy": acc,
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "ROC_AUC": roc_auc,
                "Train+Predict time (s)": elapsed,
            }
        )

        print("\n" + "-" * 60)
        print(name)
        print("Accuracy:", round(acc, 6))
        print("Precision:", round(precision, 6))
        print("Recall:", round(recall, 6))
        print("F1:", round(f1, 6))
        if is_binary and not np.isnan(roc_auc):
            print("ROC AUC:", round(roc_auc, 6))
        print("Time (s):", round(elapsed, 3))

        print(
            classification_report(
                y_test,
                preds,
                labels=label_ids,
                target_names=class_names,
                zero_division=0,
            )
        )

        print(
            "Confusion matrix:\n",
            confusion_matrix(y_test, preds, labels=label_ids)
        )

        model_file_name = sanitize_filename(name)
        cm_path = cm_dir / f"{sanitize_filename(task_name)}_{model_file_name}_confusion_matrix.png"
        save_confusion_matrix(
            y_true=y_test,
            y_pred=preds,
            class_names=class_names,
            title=f"{task_name} - confusion matrix - {name}",
            output_path=cm_path,
        )

        if is_binary and score_values is not None:
            roc_path = roc_dir / f"{sanitize_filename(task_name)}_{model_file_name}_roc.png"
            save_single_binary_roc(
                y_true=y_test,
                scores=score_values,
                model_name=name,
                output_path=roc_path,
            )

    results_df = pd.DataFrame(results).sort_values(by="F1", ascending=False)

    print("\n=== Zhrnutie modelov ===")
    print(results_df)

    if is_binary and roc_score_map:
        roc_comparison_path = task_dir / f"{sanitize_filename(task_name)}_roc_comparison.png"
        save_binary_roc_comparison(
            y_true=y_test,
            score_map=roc_score_map,
            output_path=roc_comparison_path,
        )
        print(f"Porovnávací ROC graf uložený do: {roc_comparison_path}")

    results_csv_path = task_dir / f"{sanitize_filename(task_name)}_results.csv"
    results_df.to_csv(results_csv_path, index=False)

    print(f"Tabuľka výsledkov uložená do: {results_csv_path}")
    print(f"Feature scores uložené do: {feature_scores_path}")
    print(f"Selected features uložené do: {selected_features_path}")
    print(f"Confusion matrices uložené do: {cm_dir}")
    if is_binary:
        print(f"ROC krivky uložené do: {roc_dir}")

    return results_df



# MAIN


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("BASE_DIR:", BASE_DIR)
    print("DATASET_PATH:", DATASET_PATH)
    print("OUTPUT_DIR:", OUTPUT_DIR)

    print("Načítavam dataset:", DATASET_PATH)
    df = load_dataset(DATASET_PATH)

    print("Počet záznamov:", len(df))
    print("Rozdelenie tried:\n", df[LABEL_COL].value_counts())

    X = extract_features(df[URL_COL])


    # ANALÝZA PRÍZNAKOV

    feature_analysis_dir = OUTPUT_DIR / "feature_analysis"
    feature_analysis_dir.mkdir(parents=True, exist_ok=True)

    save_numeric_feature_statistics(X, feature_analysis_dir)
    save_binary_feature_frequencies(X, feature_analysis_dir)
    plot_top_binary_feature_frequencies(X, feature_analysis_dir, top_n=15)

    selected_hist_features = [
        "url_len",
        "domain_len",
        "path_len",
        "query_len",
        "count_digits",
        "count_special",
        "digit_ratio",
        "special_ratio",
    ]
    plot_selected_numeric_histograms(X, feature_analysis_dir, selected_hist_features)

    full_corr = save_full_correlation_matrix(X, feature_analysis_dir)
    save_correlation_heatmap(full_corr, feature_analysis_dir, "full_correlation_heatmap.png")
    save_high_correlation_pairs(X, feature_analysis_dir, threshold=CORRELATION_THRESHOLD)


    # CIEĽOVÉ PREMENNÉ

    multiclass_encoder = LabelEncoder()
    y_multiclass = multiclass_encoder.fit_transform(df[LABEL_COL])
    multiclass_classes = list(multiclass_encoder.classes_)
    print("\nTriedy pre multiclass úlohu:", multiclass_classes)

    binary_labels = np.where(df[LABEL_COL].eq("benign"), "benign", "malicious")
    binary_encoder = LabelEncoder()
    y_binary = binary_encoder.fit_transform(binary_labels)
    binary_classes = [str(x) for x in list(binary_encoder.classes_)]
    print("Triedy pre binárnu úlohu:", binary_classes)


    # MODELOVANIE A VYHODNOTENIE

    multiclass_results = evaluate_task(
        task_name="Multiclass",
        X=X,
        y=y_multiclass,
        class_names=multiclass_classes,
        is_binary=False,
        output_dir=OUTPUT_DIR,
    )

    binary_results = evaluate_task(
        task_name="Binary",
        X=X,
        y=y_binary,
        class_names=binary_classes,
        is_binary=True,
        output_dir=OUTPUT_DIR,
    )

    combined = pd.concat([multiclass_results, binary_results], ignore_index=True)
    combined_path = OUTPUT_DIR / "all_results_summary.csv"
    combined.to_csv(combined_path, index=False)

    print("\nSpoločná tabuľka výsledkov uložená do:", combined_path)


if __name__ == "__main__":
    main()

BASE_DIR: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj
DATASET_PATH: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/malicious_phish.csv
OUTPUT_DIR: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs
Načítavam dataset: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/malicious_phish.csv
Počet záznamov: 641119
Rozdelenie tried:
 type
benign        428080
defacement     95308
phishing       94086
malware        23645
Name: count, dtype: int64


/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/2866479197.py:232: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "has_repeated_special": u.str.contains(repeated_special_regex, regex=True).astype(int),


Extrakcia príznakov trvala: 11.56 s
X shape: (641119, 50)
Štatistické ukazovatele príznakov uložené do: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs/feature_analysis/numeric_feature_statistics.csv
Početnosti binárnych príznakov uložené do: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs/feature_analysis/binary_feature_frequencies.csv
Graf binárnych príznakov uložený do: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs/feature_analysis/top_binary_feature_frequencies.png
Celá korelačná matica uložená do: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs/feature_analysis/full_correlation_matrix.csv
Heatmapa korelačnej matice uložená do: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs/feature_analysis/full_correlation_heatmap.png
Silno korelované páry príznakov uložené do: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs/feature_analysis/high_correlation_

In [2]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler


# MAKRO F1 GRAFY 


BASE_DIR = Path.cwd()
DATASET_PATH = BASE_DIR / "malicious_phish.csv"
OUTPUT_DIR = BASE_DIR / "bakalarka_outputs"

def vytvor_macro_f1_graf(task_name_sk, folder_name, y, is_binary):
    print(f"\nSpúšťam výpočet makro F1 pre: {task_name_sk}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    (
        X_train_sel,
        X_test_sel,
        _,
        _,
        _,
        _,
    ) = select_features(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        top_k=TOP_K,
        use_mi=USE_MI,
    )

    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_test_scaled = scaler.transform(X_test_sel)

    models = get_models(is_binary=is_binary)

    rows = []

    for name, model in models.items():
        t0 = time.perf_counter()

        if name in {"Logistic Regression", "LinearSVC"}:
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_test_scaled)
        else:
            model.fit(X_train_sel, y_train)
            preds = model.predict(X_test_sel)

        elapsed = time.perf_counter() - t0

        macro_f1 = f1_score(
            y_test,
            preds,
            average="macro",
            zero_division=0,
        )

        rows.append(
            {
                "Model": name,
                "Makro F1": macro_f1,
                "Čas tréningu a predikcie (s)": elapsed,
            }
        )

        print(f"{name}: makro F1 = {macro_f1:.4f}, čas = {elapsed:.2f} s")

    df_macro = pd.DataFrame(rows).sort_values(by="Makro F1", ascending=True)

    save_dir = OUTPUT_DIR / folder_name / "macro_f1_plots"
    save_dir.mkdir(parents=True, exist_ok=True)

    csv_path = save_dir / f"{folder_name}_macro_f1_vysledky.csv"
    df_macro.to_csv(csv_path, index=False)

    fig, ax = plt.subplots(figsize=(10, 5.5))

    bars = ax.barh(
        df_macro["Model"],
        df_macro["Makro F1"],
        height=0.6,
        color="#f9c5d1",
        edgecolor="#c96f91",
        linewidth=0.8,
    )

    ax.set_title(
        f"Porovnanie modelov podľa makro F1 skóre – {task_name_sk}",
        fontsize=15,
        fontweight="bold",
        pad=15,
    )
    ax.set_xlabel("Makro F1 skóre", fontsize=12)
    ax.set_ylabel("Klasifikačný model", fontsize=12)
    ax.set_xlim(0, 1)

    ax.grid(axis="x", linestyle="--", alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for bar in bars:
        value = bar.get_width()
        ax.text(
            value + 0.008,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}".replace(".", ","),
            va="center",
            fontsize=11,
        )

    plt.tight_layout()

    plot_path = save_dir / f"{folder_name}_macro_f1_porovnanie_modelov.png"
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"CSV uložené do: {csv_path}")
    print(f"Graf uložený do: {plot_path}")

    return df_macro



# NAČÍTANIE DÁT A EXTRAKCIA PRÍZNAKOV


df = load_dataset(DATASET_PATH)
X = extract_features(df[URL_COL])

multiclass_encoder = LabelEncoder()
y_multiclass = multiclass_encoder.fit_transform(df[LABEL_COL])

binary_labels = np.where(df[LABEL_COL].eq("benign"), "benign", "malicious")
binary_encoder = LabelEncoder()
y_binary = binary_encoder.fit_transform(binary_labels)



# VYTVORENIE GRAFOV


macro_multiclass = vytvor_macro_f1_graf(
    task_name_sk="multitriedna klasifikácia",
    folder_name="multiclass",
    y=y_multiclass,
    is_binary=False,
)

macro_binary = vytvor_macro_f1_graf(
    task_name_sk="binárna klasifikácia",
    folder_name="binary",
    y=y_binary,
    is_binary=True,
)

display(macro_multiclass.sort_values(by="Makro F1", ascending=False))
display(macro_binary.sort_values(by="Makro F1", ascending=False))

/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/2866479197.py:232: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "has_repeated_special": u.str.contains(repeated_special_regex, regex=True).astype(int),


Extrakcia príznakov trvala: 11.77 s
X shape: (641119, 50)

Spúšťam výpočet makro F1 pre: multitriedna klasifikácia
Odstránené kvôli korelácii: ['domain_without_tld_len', 'count_letters', 'letter_ratio']
Top-20 features podľa mutual_info:
count_colon            0.319222
count_double_slash     0.318816
count_slashes          0.219274
has_www                0.193269
digit_ratio            0.187380
path_len               0.164546
uppercase_ratio        0.153841
count_dots             0.151061
avg_token_len          0.127509
special_ratio          0.120837
num_subdomains         0.119067
count_special          0.111867
count_params           0.109351
num_tokens_url         0.109196
url_len                0.102356
query_len              0.101837
count_path_segments    0.092160
count_uppercase        0.084975
domain_digit_ratio     0.084180
count_digits           0.075970
dtype: float64
Logistic Regression: makro F1 = 0.7685, čas = 5.35 s
Decision Tree: makro F1 = 0.9281, čas = 1.93 s
Random 

/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/2425063335.py:129: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Odstránené kvôli korelácii: ['domain_without_tld_len', 'count_letters', 'letter_ratio']
Top-20 features podľa mutual_info:
count_double_slash     0.205559
count_colon            0.194522
has_www                0.161268
count_dots             0.126174
count_slashes          0.123274
num_subdomains         0.101398
uppercase_ratio        0.078289
path_len               0.072358
digit_ratio            0.070739
special_ratio          0.053841
count_params           0.050365
count_special          0.049011
count_uppercase        0.048229
domain_len             0.042015
avg_token_len          0.041261
query_len              0.040191
count_path_segments    0.033592
domain_digit_ratio     0.027635
has_ip                 0.018958
uses_https             0.016497
dtype: float64
Logistic Regression: makro F1 = 0.8781, čas = 0.54 s
Decision Tree: makro F1 = 0.9632, čas = 1.61 s
Random Forest: makro F1 = 0.9721, čas = 17.29 s
LinearSVC: makro F1 = 0.8861, čas = 1.48 s
XGBoost: makro F1 = 0.9692, čas

/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/2425063335.py:129: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Model,Makro F1,Čas tréningu a predikcie (s)
2,Random Forest,0.944725,21.315439
4,XGBoost,0.937456,7.133256
1,Decision Tree,0.928090,1.926329
0,Logistic Regression,0.768466,5.350276
3,LinearSVC,0.757019,9.693736


,Model,Makro F1,Čas tréningu a predikcie (s)
2,Random Forest,0.972114,17.286550
4,XGBoost,0.969170,1.789169
1,Decision Tree,0.963218,1.611247
3,LinearSVC,0.886128,1.479429
0,Logistic Regression,0.878125,0.542877


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

BASE_DIR = Path.cwd()
DATASET_PATH = BASE_DIR / "malicious_phish.csv"
OUTPUT_DIR = BASE_DIR / "bakalarka_outputs"

def vytvor_konfuznu_maticu_rf(task_name_sk, folder_name, y, class_names):
    print(f"\nVytváram konfúznu maticu pre: {task_name_sk}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    (
        X_train_sel,
        X_test_sel,
        _,
        _,
        _,
        _,
    ) = select_features(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        top_k=TOP_K,
        use_mi=USE_MI,
    )

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(X_train_sel, y_train)
    preds = model.predict(X_test_sel)

    label_ids = list(range(len(class_names)))
    cm = confusion_matrix(y_test, preds, labels=label_ids)

    save_dir = OUTPUT_DIR / folder_name / "rf_confusion_matrix_final"
    save_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(8, 6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )

    disp.plot(
        ax=ax,
        cmap="RdPu",
        colorbar=True,
        values_format="d",
    )

    ax.set_title(
        f"Konfúzna matica modelu RF – {task_name_sk}",
        fontsize=14,
        fontweight="bold",
        pad=14,
    )
    ax.set_xlabel("Predikovaná trieda", fontsize=12)
    ax.set_ylabel("Skutočná trieda", fontsize=12)

    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    output_path = save_dir / f"{folder_name}_rf_konfuzna_matica.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Konfúzna matica uložená do: {output_path}")
    print("Matica hodnôt:")
    print(cm)

    return cm


# Ak ešte nie je v pamäti df a X, tieto riadky ich pripravia:
df = load_dataset(DATASET_PATH)
X = extract_features(df[URL_COL])

# Multitriedna klasifikácia
multiclass_encoder = LabelEncoder()
y_multiclass = multiclass_encoder.fit_transform(df[LABEL_COL])
multiclass_classes = list(multiclass_encoder.classes_)

cm_multiclass_rf = vytvor_konfuznu_maticu_rf(
    task_name_sk="multitriedna klasifikácia",
    folder_name="multiclass",
    y=y_multiclass,
    class_names=multiclass_classes,
)

# Binárna klasifikácia
binary_labels = np.where(df[LABEL_COL].eq("benign"), "benign", "malicious")
binary_encoder = LabelEncoder()
y_binary = binary_encoder.fit_transform(binary_labels)
binary_classes = [str(x) for x in list(binary_encoder.classes_)]

cm_binary_rf = vytvor_konfuznu_maticu_rf(
    task_name_sk="binárna klasifikácia",
    folder_name="binary",
    y=y_binary,
    class_names=binary_classes,
)

/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/2866479197.py:232: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "has_repeated_special": u.str.contains(repeated_special_regex, regex=True).astype(int),


Extrakcia príznakov trvala: 11.63 s
X shape: (641119, 50)

Vytváram konfúznu maticu pre: multitriedna klasifikácia
Odstránené kvôli korelácii: ['domain_without_tld_len', 'count_letters', 'letter_ratio']
Top-20 features podľa mutual_info:
count_colon            0.319222
count_double_slash     0.318816
count_slashes          0.219274
has_www                0.193269
digit_ratio            0.187380
path_len               0.164546
uppercase_ratio        0.153841
count_dots             0.151061
avg_token_len          0.127509
special_ratio          0.120837
num_subdomains         0.119067
count_special          0.111867
count_params           0.109351
num_tokens_url         0.109196
url_len                0.102356
query_len              0.101837
count_path_segments    0.092160
count_uppercase        0.084975
domain_digit_ratio     0.084180
count_digits           0.075970
dtype: float64


/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/3949442813.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Konfúzna matica uložená do: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs/multiclass/rf_confusion_matrix_final/multiclass_rf_konfuzna_matica.png
Matica hodnôt:
[[84352    20     8  1236]
 [   54 18718    32   258]
 [   45   100  4298   286]
 [ 2011   459    78 16269]]

Vytváram konfúznu maticu pre: binárna klasifikácia
Odstránené kvôli korelácii: ['domain_without_tld_len', 'count_letters', 'letter_ratio']
Top-20 features podľa mutual_info:
count_double_slash     0.205559
count_colon            0.194522
has_www                0.161268
count_dots             0.126174
count_slashes          0.123274
num_subdomains         0.101398
uppercase_ratio        0.078289
path_len               0.072358
digit_ratio            0.070739
special_ratio          0.053841
count_params           0.050365
count_special          0.049011
count_uppercase        0.048229
domain_len             0.042015
avg_token_len          0.041261
query_len              0.040191
count_path_segments

/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/3949442813.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

BASE_DIR = Path.cwd()
DATASET_PATH = BASE_DIR / "malicious_phish.csv"
OUTPUT_DIR = BASE_DIR / "bakalarka_outputs"

def vytvor_konfuznu_maticu_rf(nazov_ulohy, priecinok, y, nazvy_tried):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    X_train_sel, X_test_sel, _, _, _, _ = select_features(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        top_k=TOP_K,
        use_mi=USE_MI,
    )

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(X_train_sel, y_train)
    predikcie = model.predict(X_test_sel)

    label_ids = list(range(len(nazvy_tried)))
    cm = confusion_matrix(y_test, predikcie, labels=label_ids)

    save_dir = OUTPUT_DIR / priecinok / "konfuzne_matice_rf"
    save_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(8, 6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=nazvy_tried,
    )

    disp.plot(
        ax=ax,
        cmap="RdPu",
        colorbar=True,
        values_format="d",
    )

    ax.set_title(
        f"Konfúzna matica modelu RF – {nazov_ulohy}",
        fontsize=14,
        fontweight="bold",
        pad=14,
    )
    ax.set_xlabel("Predikovaná trieda", fontsize=12)
    ax.set_ylabel("Skutočná trieda", fontsize=12)

    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    output_path = save_dir / f"{priecinok}_rf_konfuzna_matica.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Graf uložený do: {output_path}")
    print("Hodnoty konfúznej matice:")
    print(cm)

    return cm


df = load_dataset(DATASET_PATH)
X = extract_features(df[URL_COL])

# Multitriedna klasifikácia
multiclass_encoder = LabelEncoder()
y_multiclass = multiclass_encoder.fit_transform(df[LABEL_COL])
multiclass_classes = list(multiclass_encoder.classes_)

cm_multiclass_rf = vytvor_konfuznu_maticu_rf(
    nazov_ulohy="multitriedna klasifikácia",
    priecinok="multiclass",
    y=y_multiclass,
    nazvy_tried=multiclass_classes,
)

# Binárna klasifikácia
binary_labels = np.where(df[LABEL_COL].eq("benign"), "benign", "malicious")
binary_encoder = LabelEncoder()
y_binary = binary_encoder.fit_transform(binary_labels)
binary_classes = [str(x) for x in binary_encoder.classes_]

cm_binary_rf = vytvor_konfuznu_maticu_rf(
    nazov_ulohy="binárna klasifikácia",
    priecinok="binary",
    y=y_binary,
    nazvy_tried=binary_classes,
)

/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/2866479197.py:232: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "has_repeated_special": u.str.contains(repeated_special_regex, regex=True).astype(int),


Extrakcia príznakov trvala: 11.44 s
X shape: (641119, 50)
Odstránené kvôli korelácii: ['domain_without_tld_len', 'count_letters', 'letter_ratio']
Top-20 features podľa mutual_info:
count_colon            0.319222
count_double_slash     0.318816
count_slashes          0.219274
has_www                0.193269
digit_ratio            0.187380
path_len               0.164546
uppercase_ratio        0.153841
count_dots             0.151061
avg_token_len          0.127509
special_ratio          0.120837
num_subdomains         0.119067
count_special          0.111867
count_params           0.109351
num_tokens_url         0.109196
url_len                0.102356
query_len              0.101837
count_path_segments    0.092160
count_uppercase        0.084975
domain_digit_ratio     0.084180
count_digits           0.075970
dtype: float64


/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/691530767.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Graf uložený do: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs/multiclass/konfuzne_matice_rf/multiclass_rf_konfuzna_matica.png
Hodnoty konfúznej matice:
[[84352    20     8  1236]
 [   54 18718    32   258]
 [   45   100  4298   286]
 [ 2011   459    78 16269]]
Odstránené kvôli korelácii: ['domain_without_tld_len', 'count_letters', 'letter_ratio']
Top-20 features podľa mutual_info:
count_double_slash     0.205559
count_colon            0.194522
has_www                0.161268
count_dots             0.126174
count_slashes          0.123274
num_subdomains         0.101398
uppercase_ratio        0.078289
path_len               0.072358
digit_ratio            0.070739
special_ratio          0.053841
count_params           0.050365
count_special          0.049011
count_uppercase        0.048229
domain_len             0.042015
avg_token_len          0.041261
query_len              0.040191
count_path_segments    0.033592
domain_digit_ratio     0.027635
has_ip        

/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/691530767.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_curve, roc_auc_score

BASE_DIR = Path.cwd()
DATASET_PATH = BASE_DIR / "malicious_phish.csv"
OUTPUT_DIR = BASE_DIR / "bakalarka_outputs"

df = load_dataset(DATASET_PATH)
X = extract_features(df[URL_COL])

binary_labels = np.where(df[LABEL_COL].eq("benign"), "benign", "malicious")
binary_encoder = LabelEncoder()
y_binary = binary_encoder.fit_transform(binary_labels)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_binary,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_binary,
)

X_train_sel, X_test_sel, _, _, _, _ = select_features(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    top_k=TOP_K,
    use_mi=USE_MI,
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

models = get_models(is_binary=True)

plt.figure(figsize=(8, 6))

for name, model in models.items():
    if name in {"Logistic Regression", "LinearSVC"}:
        model.fit(X_train_scaled, y_train)
        scores = get_scores(model, X_test_scaled)
    else:
        model.fit(X_train_sel, y_train)
        scores = get_scores(model, X_test_sel)

    if scores is None:
        continue

    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)

    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Náhodný klasifikátor")

plt.title("ROC krivky modelov – binárna klasifikácia", fontsize=14, fontweight="bold")
plt.xlabel("Miera falošne pozitívnych predikcií")
plt.ylabel("Miera skutočne pozitívnych predikcií")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

output_path = OUTPUT_DIR / "binary" / "binary_roc_comparison.png"
output_path.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Spoločný ROC graf uložený do: {output_path}")

/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/2866479197.py:232: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "has_repeated_special": u.str.contains(repeated_special_regex, regex=True).astype(int),


Extrakcia príznakov trvala: 11.65 s
X shape: (641119, 50)
Odstránené kvôli korelácii: ['domain_without_tld_len', 'count_letters', 'letter_ratio']
Top-20 features podľa mutual_info:
count_double_slash     0.205559
count_colon            0.194522
has_www                0.161268
count_dots             0.126174
count_slashes          0.123274
num_subdomains         0.101398
uppercase_ratio        0.078289
path_len               0.072358
digit_ratio            0.070739
special_ratio          0.053841
count_params           0.050365
count_special          0.049011
count_uppercase        0.048229
domain_len             0.042015
avg_token_len          0.041261
query_len              0.040191
count_path_segments    0.033592
domain_digit_ratio     0.027635
has_ip                 0.018958
uses_https             0.016497
dtype: float64
Spoločný ROC graf uložený do: /Users/andreakravcova/Desktop/bakalarka-kravcova-maj/bakalarka_outputs/binary/binary_roc_comparison.png


/var/folders/np/tdml2lkn30l79271_df80xmh0000gn/T/ipykernel_23864/1518826820.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
